# Scope Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Sticky note vs carving.** Create a global `shop_name = "Dhaka Delights"` and a function `print_stock()` that creates its OWN local `stock = 42` and prints both names in one sentence. Call the function, then try printing `stock` outside it wrapped in `try`/`except NameError`.

In [ ]:
shop_name = "Dhaka Delights"          # global: carved into the wall


def print_stock():
    stock = 42                        # local: wiped after the call
    print(f"{shop_name} has {stock} units")


print_stock()

try:
    print(stock)                      # locals never leave the function
except NameError as err:
    print("NameError:", err)

**2. Reading yes, writing no.** With a global `count = 100`, write `show_count()` that simply prints it (this works!), and `bump_count()` that does `count += 1` (this explodes!). Call both, catching the explosion with `try`/`except UnboundLocalError` and printing the error message.

In [ ]:
count = 100


def show_count():
    print(count)              # READ: the search walks out to global


def bump_count():
    count += 1                # ASSIGN: count is now local for the WHOLE body


show_count()

try:
    bump_count()
except UnboundLocalError as err:
    print("UnboundLocalError:", err)

**3. Who sees which x.** Build the classic nesting: a global `x = "global"`, an `outer()` that sets `x = "enclosing"` and defines an `inner()` setting `x = "local"`. Inner prints its `x`; outer prints ITS `x` after calling inner; finally the module prints `x`. Write your predicted three lines in a comment BEFORE running.

In [ ]:
x = "global"


def outer():
    x = "enclosing"           # level E

    def inner():
        x = "local"           # level L wins: nearest hit stops the search
        print("inner sees:", x)

    inner()
    print("outer sees:", x)   # outer's own enclosing x


# Predicted: local x / enclosing x / global x
outer()
print("module sees:", x)      # global untouched

## Part 2 — Practice

**4. The global escape hatch.** Start with `balance = 5000`. Write `deposit_global(amount)` that declares `global balance` and adds to it, then print `balance` to see the side effect outside the function. Next write `deposit_clean(balance, amount)` that instead RECEIVES the balance and RETURNS the new one, updating `balance` with its result.

In [ ]:
balance = 5000


def deposit_global(amount):
    global balance            # opt in to mutating the module variable
    balance += amount


deposit_global(1500)
print(balance)                # 6500 -- changed OUTSIDE the function too


def deposit_clean(balance, amount):
    return balance + amount   # state in, new state out


balance = deposit_clean(balance, 2000)
print(balance)                # 8500

**5. Counter with a memory.** Write `make_counter()` holding `hits = 0` with an inner `tick()` that declares `nonlocal hits`, increments it and returns it. Calling the returned function three times should print `1 2 3`. Then write `make_counter_broken()` WITHOUT the declaration, catch its `UnboundLocalError`, and print the error.

In [ ]:
def make_counter():
    hits = 0                  # lives in make_counter's scope

    def tick():
        nonlocal hits         # update the ENCLOSING variable
        hits += 1
        return hits

    return tick


page_views = make_counter()
print(page_views(), page_views(), page_views())   # 1 2 3


def make_counter_broken():
    hits = 0

    def tick():
        hits += 1             # no declaration -> local to tick -> boom
        return hits

    return tick


broken = make_counter_broken()
try:
    broken()
except UnboundLocalError as err:
    print("UnboundLocalError:", err)

**6. Peek at the namespaces.** Write a function `peek()` that defines `topic = "scope"` and prints `sorted(locals())`. After calling it, check whether `"topic"` and `"peek"` are keys of `globals()` and print both booleans.

In [ ]:
def peek():
    topic = "scope"
    print(sorted(locals()))       # ['topic']


peek()

print("'topic' in globals():", "topic" in globals())
print("'peek' in globals():", "peek" in globals())

**7. No fences around blocks.** Loop `for i in range(3):` with `doubled = i * 2` inside. After the loop ends, print BOTH `i` and `doubled`. Python has no block scope — confirm both variables survived.

In [ ]:
for i in range(3):
    doubled = i * 2

print(i)         # 2 -- survived the loop
print(doubled)   # 4 -- so did this

## Part 3 — Challenge

**8. Private piggy banks.** Write `make_account(start)` returning an inner `spend(amount)` that keeps its own `balance` updated via `nonlocal` and returns the money left. Open Sarah's account with 1000 and Rahim's with 50, spend 300 and 20 respectively, then another 100 from Sarah's. The expected prints are `700`, `30`, `600` — prove Rahim's money never mixed with Sarah's.

In [ ]:
def make_account(start):
    balance = start           # captured: one cell per account

    def spend(amount):
        nonlocal balance
        balance -= amount
        return balance

    return spend


sarah_acct = make_account(1000)
rahim_acct = make_account(50)     # completely separate closure cell

print(sarah_acct(300))            # 700
print(rahim_acct(20))             # 30
print(sarah_acct(100))            # 600 -- untouched by Rahim